## Setup

In [4]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [5]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [ ]:
# from ner import evaluation

In [6]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [7]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### wikiann

In [8]:
# from ner.dataset_base import HuggingFaceMultilingualDataset
# class Wikiann(HuggingFaceMultilingualDataset):
#     dataset_name = 'wikiann'
#     language = 'ro'
#     license = 'unknown'

# dataset = Wikiann()
# dataset.check_labels()

In [9]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset('wikiann', wikiann_label_map, lang='ro')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

In [10]:
print(ner.check_labels(wikiann_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'B-PER', 'I-PER', 'B-LOC', 'I-ORG', 'O', 'I-LOC', 'B-ORG'}


### Ronec
https://huggingface.co/datasets/ronec


In [11]:
ronec_label_map = {
    'O': 0,
    'B-PERSON': 1,
    'I-PERSON': 2,
    'B-GPE': 3,
    'I-GPE': 4,
    'B-LOC': 5,
    'I-LOC': 6,
    'B-ORG': 7,
    'I-ORG': 8,
    'B-LANGUAGE': 9,
    'I-LANGUAGE': 10,
    'B-NAT_REL_POL': 11 ,
    'I-NAT_REL_POL': 12,
    'B-DATETIME': 13,
    'I-DATETIME': 14,
    'B-PERIOD': 15,
    'I-PERIOD': 16,
    'B-QUANTITY': 17,
    'I-QUANTITY': 18,
    'B-MONEY': 19,
    'I-MONEY': 20,
    'B-NUMERIC': 21,
    'I-NUMERIC': 22,
    'B-ORDINAL': 23,
    'I-ORDINAL': 24,
    'B-FACILITY': 25,
    'I-FACILITY': 26,
    'B-WORK_OF_ART': 27 ,
    'I-WORK_OF_ART': 28,
    'B-EVENT': 29,
    'I-EVENT': 30,
}

ronec = ner.ReadNERData()
ronec_words, ronec_labels = ronec.read_dataset('ronec', ronec_label_map)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test Split


  0%|          | 0/2000 [00:00<?, ?it/s]

In [12]:
print(ner.check_labels(ronec_labels))
label_alignment = {
    'O': 'O',
    'B-PERSON': 'B-PER',
    'I-PERSON': 'I-PER',
    'B-GPE': 'O',
    'I-GPE': 'O',
    'B-LOC': 'B-LOC',
    'I-LOC': 'I-LOC',
    'B-ORG': 'B-ORG',
    'I-ORG': 'I-ORG',
    'B-LANGUAGE': 'O',
    'I-LANGUAGE': 'O',
    'B-NAT_REL_POL': 'O',
    'I-NAT_REL_POL': 'O',
    'B-DATETIME': 'O',
    'I-DATETIME': 'O',
    'B-PERIOD': 'O',
    'I-PERIOD': 'O',
    'B-QUANTITY': 'O',
    'I-QUANTITY': 'O',
    'B-MONEY': 'O',
    'I-MONEY': 'O',
    'B-NUMERIC': 'O',
    'I-NUMERIC': 'O',
    'B-ORDINAL': 'O',
    'I-ORDINAL': 'O',
    'B-FACILITY': 'O',
    'I-FACILITY': 'O',
    'B-WORK_OF_ART': 'O',
    'I-WORK_OF_ART': 'O',
    'B-EVENT': 'O',
    'I-EVENT': 'O',
}
# Align the dataset labels to the standard labels
ronec_labels = ner.align_dataset(ronec_labels, label_alignment)
print(ner.check_labels(ronec_labels))

{'I-WORK_OF_ART', 'I-PERSON', 'I-GPE', 'B-LANGUAGE', 'B-QUANTITY', 'O', 'I-ORG', 'B-PERIOD', 'B-ORG', 'B-NUMERIC', 'I-NAT_REL_POL', 'I-MONEY', 'I-FACILITY', 'I-EVENT', 'B-NAT_REL_POL', 'I-LOC', 'B-GPE', 'B-EVENT', 'B-MONEY', 'B-PERSON', 'I-LANGUAGE', 'B-DATETIME', 'I-DATETIME', 'I-ORDINAL', 'I-QUANTITY', 'I-NUMERIC', 'B-LOC', 'B-WORK_OF_ART', 'B-ORDINAL', 'I-PERIOD', 'B-FACILITY'}
{'B-PER', 'I-PER', 'B-LOC', 'I-ORG', 'O', 'I-LOC', 'B-ORG'}


# Evaluate model

In [17]:
alignment = {
'LABEL_0': 'O',
'LABEL_1': 'B-PER',
'LABEL_2': 'I-PER',
'LABEL_3': 'B-ORG',
'LABEL_4': 'I-ORG',
'LABEL_5': 'O',
'LABEL_6': 'O',
'LABEL_7': 'B-LOC',
'LABEL_8': 'I-LOC',
'LABEL_9': 'O',
'LABEL_10': 'O',
'LABEL_11': 'O',
'LABEL_12': 'O',
'LABEL_13': 'O',
'LABEL_14': 'O',
'LABEL_15': 'O',
'LABEL_16': 'O',
'LABEL_17': 'O',
'LABEL_18': 'O',
'LABEL_19': 'O',
'LABEL_20': 'O',
'LABEL_21': 'O',
'LABEL_22': 'O',
'LABEL_23': 'O',
'LABEL_24': 'O',
'LABEL_25': 'O',
'LABEL_26': 'O',
'LABEL_27': 'O',
'LABEL_28': 'O',
'LABEL_29': 'O',
'LABEL_30': 'O',
 }

model_name = "dumitrescustefan/bert-base-romanian-ner"
model_name_output = 'dumitrescustefan/bert-base-romanian-ner'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [14]:
model_evaluation.model.config.id2label

{0: 'LABEL_0',
 1: 'LABEL_1',
 2: 'LABEL_2',
 3: 'LABEL_3',
 4: 'LABEL_4',
 5: 'LABEL_5',
 6: 'LABEL_6',
 7: 'LABEL_7',
 8: 'LABEL_8',
 9: 'LABEL_9',
 10: 'LABEL_10',
 11: 'LABEL_11',
 12: 'LABEL_12',
 13: 'LABEL_13',
 14: 'LABEL_14',
 15: 'LABEL_15',
 16: 'LABEL_16',
 17: 'LABEL_17',
 18: 'LABEL_18',
 19: 'LABEL_19',
 20: 'LABEL_20',
 21: 'LABEL_21',
 22: 'LABEL_22',
 23: 'LABEL_23',
 24: 'LABEL_24',
 25: 'LABEL_25',
 26: 'LABEL_26',
 27: 'LABEL_27',
 28: 'LABEL_28',
 29: 'LABEL_29',
 30: 'LABEL_30'}

In [16]:
 print(['O', 'B-PERSON', 'I-PERSON', 'B-ORG', 'I-ORG', 'B-GPE', 'I-GPE', 'B-LOC', 'I-LOC', 'B-NAT_REL_POL', 'I-NAT_REL_POL', 'B-EVENT', 'I-EVENT', 'B-LANGUAGE', 'I-LANGUAGE', 'B-WORK_OF_ART', 'I-WORK_OF_ART', 'B-DATETIME', 'I-DATETIME', 'B-PERIOD', 'I-PERIOD', 'B-MONEY', 'I-MONEY', 'B-QUANTITY', 'I-QUANTITY', 'B-NUMERIC', 'I-NUMERIC', 'B-ORDINAL', 'I-ORDINAL', 'B-FACILITY', 'I-FACILITY'])

['O', 'B-PERSON', 'I-PERSON', 'B-ORG', 'I-ORG', 'B-GPE', 'I-GPE', 'B-LOC', 'I-LOC', 'B-NAT_REL_POL', 'I-NAT_REL_POL', 'B-EVENT', 'I-EVENT', 'B-LANGUAGE', 'I-LANGUAGE', 'B-WORK_OF_ART', 'I-WORK_OF_ART', 'B-DATETIME', 'I-DATETIME', 'B-PERIOD', 'I-PERIOD', 'B-MONEY', 'I-MONEY', 'B-QUANTITY', 'I-QUANTITY', 'B-NUMERIC', 'I-NUMERIC', 'B-ORDINAL', 'I-ORDINAL', 'B-FACILITY', 'I-FACILITY']


### wikiann

In [18]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [19]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.2215,0.0266,0.0474,3804
1,ORG,0.4843,0.2657,0.3431,3666
2,PER,0.5523,0.7441,0.6340,4220
3,micro,0.5171,0.3606,0.4249,11690
4,macro,0.4194,0.3454,0.3415,11690
5,weighted,0.4234,0.3606,0.3519,11690


In [20]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.5413,0.0620,0.1113,3804
1,B-ORG,0.7176,0.3598,0.4793,3666
2,B-PER,0.6786,0.8851,0.7682,4220
3,I-LOC,0.6535,0.0371,0.0701,7583
4,I-ORG,0.6870,0.3292,0.4451,10104
5,I-PER,0.8727,0.7960,0.8326,8314
6,O,0.5677,0.9017,0.6967,28991
7,accuracy,0.6247,66682,None,None
8,macro,0.6741,0.4815,0.4862,66682
9,weighted,0.6473,0.6247,0.5635,66682


### Ronec

In [21]:
data_name = "ronec"
ronec_evaluation_output = model_evaluation.evaluate_model(ronec_words, ronec_labels)

  0%|          | 0/125 [00:00<?, ?it/s]

In [22]:
ronec_seqeval = ronec_evaluation_output.get_classification('Seqeval')
ronec_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.0214,0.0046,0.0076,1728
1,ORG,0.0000,0.0000,0.0000,373
2,PER,0.9570,0.9775,0.9671,4230
3,micro,0.6864,0.6544,0.6700,6331
4,macro,0.3261,0.3274,0.3249,6331
5,weighted,0.6452,0.6544,0.6483,6331


In [23]:
ronec_sklearn = ronec_evaluation_output.get_classification('Sklearn')
ronec_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.0216,0.0046,0.0076,1728
1,B-ORG,0.0000,0.0000,0.0000,373
2,B-PER,0.9708,0.9903,0.9805,4230
3,I-LOC,0.0000,0.0000,0.0000,273
4,I-ORG,0.0000,0.0000,0.0000,503
5,I-PER,0.9795,0.9822,0.9808,2186
6,O,0.9736,0.9536,0.9635,79176
7,accuracy,0.9251,88469,None,None
8,macro,0.4208,0.4187,0.4189,88469
9,weighted,0.9424,0.9251,0.9335,88469
